In [1]:
print("Let's Start")

Let's Start


In [2]:
import pandas as td
from sklearn.model_selection import train_test_split
df = td.read_csv('/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv' , index_col=0)

df = df.iloc[: , :2]

x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
y = labels.map(label_mapping)
if y.isna().any():
    invalid = sorted(labels[y.isna()].unique())
    raise ValueError(f"Unsupported categories: {invalid}")

x_train_, x_test_, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )


In [4]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

x_train_tf = vectorizer.fit_transform(x_train_)
x_test_tf = vectorizer.transform(x_test_)

model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(x_train_tf, y_train)
y_pred = model.predict(x_test_tf)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.548036 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15402
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4344
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
              precision    recall  f1-score   support

           0       0.85      0.97      0.91      2529
           1       0.92      0.85      0.88      3154
           2       0.82      0.78      0.80      1650

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.86      7333
weighted avg       0.87      0.87      0.87      7333

Confusion Matrix:
 [[2449   33   47]
 [ 249 2672  233]
 [ 170  199 1281]]


In [5]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)

x = vectorizer.fit_transform(x)
smote = SMOTE(random_state=42)
x , y = smote.fit_resample(x , y)
x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.130746 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 59066
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 5187
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
              precision    recall  f1-score   support

           0       0.88      0.97      0.93      3154
           1       0.91      0.88      0.89      3154
           2       0.93      0.86      0.89      3154

    accuracy                           0.90      9462
   macro avg       0.91      0.90      0.90      9462
weighted avg       0.91      0.90      0.90      9462

Confusion Matrix:
 [[3069   37   48]
 [ 210 2774  170]
 [ 199  235 2720]]
